In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import time # To time operations

# Scikit-learn imports
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import GridSearchCV, KFold, cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_absolute_error, mean_squared_error

# --- Import the data loader from your APT script ---
try:
    import importlib
    import GBR_APT_optimized # Import the APT-specific script
    importlib.reload(GBR_APT_optimized)
    from GBR_APT_optimized import load_and_merge_datasets_apt # Import the APT-specific function
except ImportError:
    print("Error: Could not import from GBR_APT_optimized.py.")
    print("Make sure the file is in the same directory as the notebook.")
    load_and_merge_datasets_apt = None
except Exception as e:
    print(f"An error occurred during import: {e}")
    load_and_merge_datasets_apt = None

# --- Configuration ---
# <<< IMPORTANT: UPDATE THESE PATHS FOR APT DATA >>>
session1_path = 'Model1 APT/study2_associative_session_1_data_transformed.csv' # <--- CHANGE THIS
session2_path = 'Model1 APT/study2_associative_session_2_data_transformed.csv' # <--- CHANGE THIS

# <<< SET YOUR DESIRED DATASET NAME MANUALLY HERE >>>
manual_dataset_name = "APT Model" # <--- CHANGE THIS AS NEEDED (e.g., "APT Model 1")

# Define plot saving directory using the manual name
plot_save_dir = f"plots_GBR_CV_{manual_dataset_name.replace(' ', '_')}"
print(f"Dataset Name: {manual_dataset_name}")
print(f"Plots will be saved to: {plot_save_dir}")
os.makedirs(plot_save_dir, exist_ok=True) # Create dir if it doesn't exist

# Analysis parameters
target_col = 'alpha_s2' # Assuming alpha is still the target
random_seed = 42
n_cv_folds = 5 # Number of folds for cross-validation
n_repeats_perm_imp = 10 # Number of repeats for permutation importance

# --- Define feature sets specifically for APT ---
# List all potential APT features (after renaming in the load function)
all_possible_features_apt = [
    'alpha_s1',
    
    'a_s1', 'a_s2',
    'ndt1_s1', 'ndt1_s2', 'ndt2_s1', 'ndt2_s2', 'ndt3_s1', 'ndt3_s2', 'ndt4_s1', 'ndt4_s2',
    
]

# Define the specific feature combinations (conditions) you want to test for APT
# ADJUST THESE EXAMPLES TO YOUR SPECIFIC RESEARCH QUESTIONS FOR APT
feature_sets = {
    "All_Controls_APT": all_possible_features_apt, # Model with alpha_s1 + all other params
     "Threshold_ZR_APT": ['alpha_s1', 'a_s1', 'a_s2', 'zr_s1', 'zr_s2'],
     "NDTs_APT": ['alpha_s1', 'ndt1_s1', 'ndt1_s2', 'ndt2_s1', 'ndt2_s2', 'ndt3_s1', 'ndt3_s2', 'ndt4_s1', 'ndt4_s2'],
     "Alpha_Only_APT": ["alpha_s1"] # Baseline model
}


# Define simplified hyperparameter grid for GBR
param_grid_gbr = {
    'n_estimators': [100, 200],
    'max_depth': [2],
    'learning_rate': [0.05, 0.1],
    'min_samples_leaf': [5, 10]
}

# Define scoring metrics for cross-validation
scoring_metrics = {
    'r2': make_scorer(r2_score),
    'neg_MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'neg_RMSE': make_scorer(mean_squared_error, squared=False, greater_is_better=False)
}

# --- Load APT Data ---
if load_and_merge_datasets_apt:
    try:
        # Use the APT-specific loading function
        df_merged = load_and_merge_datasets_apt(session1_path, session2_path)
        if df_merged is not None and not df_merged.empty:
             print(f"\nAPT Data loaded. Final N = {len(df_merged)}")
             # Verify target column exists
             if target_col not in df_merged.columns:
                 raise ValueError(f"Target column '{target_col}' not found in loaded APT data.")
             y = df_merged[target_col]
        else:
             print("Loaded APT data is empty or None.")
             df_merged = None # Ensure df_merged is None if loading failed

    except FileNotFoundError:
        print(f"\nError: One or both APT input files not found.")
        df_merged = None
    except ValueError as ve:
         print(f"\nError loading APT data: {ve}")
         df_merged = None
    except Exception as e:
        print(f"\nAn unexpected error occurred during APT data loading: {e}")
        df_merged = None
else:
    print("APT data loading function not imported.")
    df_merged = None

# --- Run Analysis for Each Condition ---
results_cv = {} # Store CV results

if df_merged is not None:
    for condition_name, current_features in feature_sets.items():
        print("\n" + "="*80)
        print(f"--- Running CV Analysis for Condition: {condition_name} ---")
        print(f"Features: {current_features}")
        print("="*80)

        # Verify all features for *this specific condition* exist in the loaded data
        missing_features = [feat for feat in current_features if feat not in df_merged.columns]
        if missing_features:
             print(f"Skipping condition '{condition_name}' due to missing feature columns: {missing_features}")
             continue

        X = df_merged[current_features]
        feature_names_list = X.columns.tolist() # Use actual columns from X

        start_time = time.time()

        # --- Cross-Validation for Performance Estimation ---
        print(f"Performing {n_cv_folds}-fold Cross-Validation with GridSearchCV...")
        gbr = GradientBoostingRegressor(random_state=random_seed)
        print("Running GridSearchCV on full data to find best params...")
        gbr_grid_search = GridSearchCV(gbr, param_grid=param_grid_gbr, cv=n_cv_folds, scoring='r2', n_jobs=-1)
        gbr_grid_search.fit(X, y)
        best_gbr_model = gbr_grid_search.best_estimator_
        print(f"Best GBR Params found: {gbr_grid_search.best_params_}")
        print(f"\nRunning {n_cv_folds}-fold cross-validation using best estimator...")
        cv_results = cross_validate(best_gbr_model, X, y,
                                    cv=KFold(n_splits=n_cv_folds, shuffle=True, random_state=random_seed),
                                    scoring=scoring_metrics,
                                    n_jobs=-1,
                                    return_train_score=True)
        results_cv[condition_name] = cv_results # Store raw CV results

        print("\n--- Cross-Validation Performance ---")
        mean_test_r2 = np.mean(cv_results['test_r2'])
        std_test_r2 = np.std(cv_results['test_r2'])
        mean_test_mae = -np.mean(cv_results['test_neg_MAE'])
        std_test_mae = np.std(cv_results['test_neg_MAE'])
        mean_test_rmse = -np.mean(cv_results['test_neg_RMSE'])
        std_test_rmse = np.std(cv_results['test_neg_RMSE'])
        print(f"Mean Test R2  : {mean_test_r2:.4f} (+/- {std_test_r2:.4f})")
        print(f"Mean Test MAE : {mean_test_mae:.4f} (+/- {std_test_mae:.4f})")
        print(f"Mean Test RMSE: {mean_test_rmse:.4f} (+/- {std_test_rmse:.4f})")
        mean_train_r2 = np.mean(cv_results['train_r2'])
        print(f"(Mean Train R2: {mean_train_r2:.4f})")

        # --- Permutation Importance on Final Model ---
        print("\nTraining final model on all data for importance/PDP...")
        final_model = gbr_grid_search.best_estimator_
        print("Calculating permutation importance...")
        # Use X (current features for this condition) for importance calculation
        perm_imp = permutation_importance(final_model, X, y,
                                          n_repeats=n_repeats_perm_imp,
                                          random_state=random_seed,
                                          n_jobs=-1)

        print("\n--- Permutation Importance (on full data model) ---")
        sorted_idx = perm_imp.importances_mean.argsort()[::-1]
        print("(Higher is more important)")
        importance_data = {}
        importance_std_data = {}
        # Use feature_names_list (actual features in X for this condition)
        for i in range(len(feature_names_list)):
             feature_name = feature_names_list[i]
             mean_imp = perm_imp.importances_mean[i]
             std_imp = perm_imp.importances_std[i]
             importance_data[feature_name] = mean_imp
             importance_std_data[feature_name] = std_imp
        # Print sorted importance
        for i in sorted_idx:
             feature_name = feature_names_list[i]
             print(f"  {feature_name:<12}: Mean Importance = {importance_data[feature_name]:.4f} (+/- {importance_std_data[feature_name]:.4f})")

        # Store importance dicts in results_cv
        if condition_name in results_cv:
             results_cv[condition_name]['importance'] = importance_data
             results_cv[condition_name]['importance_std'] = importance_std_data

        # --- Plot Partial Dependence for alpha_s1 (using final model) ---
        print("\n--- Partial Dependence Plot for alpha_s1 ---")
        if 'alpha_s1' in feature_names_list:
            try:
                fig_pdp, ax_pdp = plt.subplots(figsize=(8, 6))
                PartialDependenceDisplay.from_estimator(
                    final_model, X, features=[feature_names_list.index('alpha_s1')],
                    feature_names=feature_names_list, ax=ax_pdp,
                    line_kw={"linewidth": 2.5, "color": "darkorange"}
                )
                main_title = f"PDP: Predicted {target_col} vs. alpha_s1"
                plot_title = f"{main_title}\\nDataset: {manual_dataset_name} (Cond: {condition_name})"
                ax_pdp.set_title(plot_title, fontsize=16)
                ax_pdp.set_xlabel('alpha_s1', fontsize=14) # Corrected font size
                ax_pdp.set_ylabel(f"Effect on Predicted {target_col}", fontsize=14) # Corrected font size
                ax_pdp.tick_params(axis='both', which='major', labelsize=12) # Corrected font size
                ax_pdp.grid(True, linestyle='--', alpha=0.7)
                plt.tight_layout()
                safe_suffix = "".join(c if c.isalnum() else "_" for c in f"_{condition_name}")
                filename = f"PDP_alpha_s1{safe_suffix}.png"
                filepath = os.path.join(plot_save_dir, filename)
                try:
                    plt.savefig(filepath)
                    print(f"Saved PDP plot to: {filepath}")
                except Exception as save_err:
                    print(f"Error saving PDP plot to {filepath}: {save_err}")
                plt.show()
                plt.close(fig_pdp)
            except Exception as plot_err:
                 print(f"Error generating PDP plot for alpha_s1: {plot_err}")
                 import traceback
                 traceback.print_exc()
        else:
             print("alpha_s1 not in features for this condition.")

        end_time = time.time()
        print(f"--- Completed condition '{condition_name}' in {end_time - start_time:.2f} seconds ---")


    # --- Create and Save Comparison Table ---
    print("\n" + "="*80)
    print("--- Creating and Saving Comparison Table ---")
    print("="*80)
    comparison_data = []

    # Define the order of columns for the final APT table
    column_order = [
        'Condition', 'Mean Test R2', 'Std Test R2', 'Mean Test MAE', 'Std Test MAE',
        'Mean Test RMSE', 'Std Test RMSE'
    ]
    # Add importance columns for all possible APT features
    for feat in all_possible_features_apt: # USE THE APT LIST
        column_order.append(f'Imp_{feat}')
        column_order.append(f'ImpStd_{feat}')


    for condition, res in results_cv.items():
        row_data = {'Condition': condition}

        # Get performance metrics
        row_data['Mean Test R2'] = np.mean(res.get('test_r2', [np.nan]))
        row_data['Std Test R2'] = np.std(res.get('test_r2', [np.nan]))
        row_data['Mean Test MAE'] = -np.mean(res.get('test_neg_MAE', [np.nan]))
        row_data['Std Test MAE'] = np.std(res.get('test_neg_MAE', [np.nan]))
        row_data['Mean Test RMSE'] = -np.mean(res.get('test_neg_RMSE', [np.nan]))
        row_data['Std Test RMSE'] = np.std(res.get('test_neg_RMSE', [np.nan]))

        # Get importance metrics for all possible APT features
        imp_dict = res.get('importance', {})
        imp_std_dict = res.get('importance_std', {})
        for feat in all_possible_features_apt: # USE THE APT LIST
            row_data[f'Imp_{feat}'] = imp_dict.get(feat, np.nan)
            row_data[f'ImpStd_{feat}'] = imp_std_dict.get(feat, np.nan)

        comparison_data.append(row_data)

    df_comparison = pd.DataFrame(comparison_data)
    # Reorder columns, handle potential missing columns gracefully
    df_comparison = df_comparison.reindex(columns=column_order, fill_value=np.nan)

    # Display the table in the notebook
    print("\nComparison Table:")
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    display(df_comparison.round(4))

    # Save the table to CSV
    comparison_csv_filename = f"comparison_GBR_CV_{manual_dataset_name.replace(' ', '_')}.csv"
    try:
        df_comparison.to_csv(comparison_csv_filename, index=False, float_format='%.4f')
        print(f"\nComparison table saved to: {comparison_csv_filename}")
    except Exception as e:
        print(f"\nError saving comparison table to CSV: {e}")

else:
    print("\nAnalysis prerequisites not met (data loading failed).")

Dataset Name: APT Model
Plots will be saved to: plots_GBR_CV_APT_Model
Loading APT Session 1: Model1 APT/study2_associative_session_1_data_transformed.csv
Loading APT Session 2: Model1 APT/study2_associative_session_2_data_transformed.csv
Using participant ID column: ID
Merging on 'ID'. Initial S1 rows: 128, S2 rows: 128
Rows after inner merge: 128
Keeping columns: ['ID', 'alpha_s1', 'alpha_s2', 'v1_s1', 'v1_s2', 'v2_s1', 'v2_s2', 'v3_s1', 'v3_s2', 'v4_s1', 'v4_s2', 'zr_s1', 'zr_s2', 'a_s1', 'a_s2', 'ndt1_s1', 'ndt1_s2', 'ndt2_s1', 'ndt2_s2', 'ndt3_s1', 'ndt3_s2', 'ndt4_s1', 'ndt4_s2', 'sndt_s1', 'sndt_s2']
Shape before dropping NaNs in essential APT columns: (128, 27)
Shape after dropping NaNs in essential APT columns: (128, 27)
--- APT Data Loading and Merging Complete ---

APT Data loaded. Final N = 128

--- Running CV Analysis for Condition: All_Controls_APT ---
Features: ['alpha_s1', 'a_s1', 'a_s2', 'ndt1_s1', 'ndt1_s2', 'ndt2_s1', 'ndt2_s2', 'ndt3_s1', 'ndt3_s2', 'ndt4_s1', 'ndt4

ValueError: 
All the 40 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
24 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/myenv/lib/python3.8/site-packages/sklearn/model_selection/_validation.py", line 729, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/opt/anaconda3/envs/myenv/lib/python3.8/site-packages/sklearn/base.py", line 1152, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/opt/anaconda3/envs/myenv/lib/python3.8/site-packages/sklearn/ensemble/_gb.py", line 424, in fit
    y = column_or_1d(y, warn=True)
  File "/opt/anaconda3/envs/myenv/lib/python3.8/site-packages/sklearn/utils/validation.py", line 1244, in column_or_1d
    raise ValueError(
ValueError: y should be a 1d arra